#### Load Product Information

Load the product dataset to examine whether product categories and brands influence the relationship between discounts and purchase quantities.

In [1]:
import pandas as pd

In [2]:
product = pd.read_csv("../data/raw/product.csv")

print("Product dataset shape:", product.shape)
print("\nColumns:")
print(product.columns.tolist())

product.head()

Product dataset shape: (92353, 7)

Columns:
['PRODUCT_ID', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']


,PRODUCT_ID,MANUFACTURER,DEPARTMENT,BRAND,COMMODITY_DESC,SUB_COMMODITY_DESC,CURR_SIZE_OF_PRODUCT
0,25671,2,GROCERY,National,FRZN ICE,ICE - CRUSHED/CUBED,22 LB
1,26081,2,MISC. TRANS.,National,NO COMMODITY DESCRIPTION,NO SUBCOMMODITY DESCRIPTION,
2,26093,69,PASTRY,Private,BREAD,BREAD:ITALIAN/FRENCH,
3,26190,69,GROCERY,Private,FRUIT - SHELF STABLE,APPLE SAUCE,50 OZ
4,26355,69,GROCERY,Private,COOKIES/CONES,SPECIALTY COOKIES,14 OZ


#### Identify Product Categories

Identify the available product categories and brand information for grouping transactions.

In [3]:
category_cols = [
    col for col in product.columns
    if any(x in col.lower() for x in ["category", "department", "brand"])
]

product[category_cols].nunique()

DEPARTMENT    44
BRAND          2
dtype: int64

#### Inspect Product Commodity Groups

Check the number and quality of commodity categories before merging them with transaction records.

In [4]:
print("Unique commodities:", product["COMMODITY_DESC"].nunique())
print("Unique sub-commodities:", product["SUB_COMMODITY_DESC"].nunique())

print("\nMissing commodity descriptions:")
print(product["COMMODITY_DESC"].isna().sum())

print("\nPlaceholder commodity descriptions:")
print((product["COMMODITY_DESC"] == "NO COMMODITY DESCRIPTION").sum())

print("\nMost common commodities:")
product["COMMODITY_DESC"].value_counts().head(10)

Unique commodities: 308
Unique sub-commodities: 2383

Missing commodity descriptions:
0

Placeholder commodity descriptions:
490

Most common commodities:


COMMODITY_DESC
GREETING CARDS/WRAP/PARTY SPLY    2785
CANDY - PACKAGED                  2475
MAKEUP AND TREATMENT              2467
HAIR CARE PRODUCTS                1744
SOFT DRINKS                       1704
BAG SNACKS                        1523
HISPANIC                          1460
FRZN MEAT/MEAT DINNERS            1268
STATIONERY & SCHOOL SUPPLIES      1261
MAGAZINE                          1224
Name: count, dtype: int64

#### Validate Product IDs

Check whether each product ID maps to exactly one product record before merging.

In [5]:
print("Total product records:", len(product))
print("Unique product IDs:", product["PRODUCT_ID"].nunique())
print("Duplicate product IDs:", product["PRODUCT_ID"].duplicated().sum())

Total product records: 92353
Unique product IDs: 92353
Duplicate product IDs: 0


#### Merge Product Information

Merge product category and brand details with transaction data using PRODUCT_ID.

In [6]:
transactions_promo = pd.read_csv("../data/processed/transactions_promo.csv")

print("Dataset shape:", transactions_promo.shape)
transactions_promo.head()

Dataset shape: (2595732, 21)


/tmp/ipykernel_2610488/617246992.py:1: DtypeWarning: Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions_promo = pd.read_csv("../data/processed/transactions_promo.csv")


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,...,COUPON_MATCH_DISC,display_values,mailer_values,promotion_records,retail_discount_amount,coupon_discount_amount,coupon_match_discount_amount,total_discount,estimated_original_value,discount_percentage
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,...,0.0,NaN,NaN,NaN,0.60,-0.0,-0.0,0.60,1.99,30.150754
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,...,0.0,NaN,NaN,NaN,-0.00,-0.0,-0.0,0.00,0.82,0.000000
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,...,0.0,NaN,NaN,NaN,0.30,-0.0,-0.0,0.30,1.29,23.255814
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,...,0.0,NaN,NaN,NaN,-0.00,-0.0,-0.0,0.00,1.21,0.000000
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,...,0.0,NaN,NaN,NaN,0.39,-0.0,-0.0,0.39,1.89,20.634921


In [7]:
product_info = product[
    ["PRODUCT_ID", "COMMODITY_DESC", "SUB_COMMODITY_DESC", "BRAND"]
]

transactions_enriched = transactions_promo.merge(
    product_info,
    on="PRODUCT_ID",
    how="left",
    validate="many_to_one"
)

print("Original transaction rows:", len(transactions_promo))
print("After product merge:", len(transactions_enriched))
print(
    "Missing commodity matches:",
    transactions_enriched["COMMODITY_DESC"].isna().sum()
)

Original transaction rows: 2595732
After product merge: 2595732
Missing commodity matches: 0


#### Save Enriched Transaction Data

Save the transaction dataset with additional product information for further analysis.

In [8]:
transactions_enriched.to_csv(
    "../data/processed/transactions_enriched.csv",
    index=False
)